<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [4]:
using System;
using System.Collections.Generic;
using System.Linq;

// ========== ДЕЛЕГАТЫ ==========
public delegate void PersonEventHandler(string message);
public delegate void GradeChangeHandler(string studentName, double newAverage);
public delegate void SalaryChangeHandler(string employeeName, decimal newSalary);

// ========== БАЗОВЫЙ КЛАСС PERSON ==========
public abstract class Person
{
    // Свойства
    public string Name { get; set; }
    public int Age { get; set; }
    public string Gender { get; set; }
    public string Email { get; set; }
    public string Phone { get; set; }
    
    // Статическая коллекция для хранения всех людей
    public static List<Person> AllPeople { get; private set; } = new List<Person>();
    
    // Статическое событие
    public static event PersonEventHandler PersonCreated;
    public event PersonEventHandler PersonUpdated;
    public event PersonEventHandler PersonDeleted;
    
    public Person(string name, int age, string gender)
    {
        Name = name;
        Age = age;
        Gender = gender;
        AllPeople.Add(this);
        OnPersonCreated($"Создан {GetType().Name}: {Name}");
    }
    
    public Person(string name, int age, string gender, string email) : this(name, age, gender)
    {
        Email = email;
    }
    
    public Person(string name, int age, string gender, string email, string phone) : this(name, age, gender, email)
    {
        Phone = phone;
    }
    
    // Защищенные методы для вызова событий
    protected virtual void OnPersonCreated(string message)
    {
        PersonCreated?.Invoke(message);
    }
    
    protected virtual void OnPersonUpdated(string message)
    {
        PersonUpdated?.Invoke(message);
    }
    
    protected virtual void OnPersonDeleted(string message)
    {
        PersonDeleted?.Invoke(message);
    }
    
    // Виртуальные методы
    public virtual string GetInfo()
    {
        return $"Имя: {Name}, Возраст: {Age}, Пол: {Gender}";
    }
    
    public virtual void SayHello()
    {
        Console.WriteLine($"Привет! Меня зовут {Name}.");
    }
    
    public abstract void Work();
    public abstract void Study();
    public abstract void Relax();
    
    // Метод для обновления информации
    public void UpdateInfo(string name, int age, string email = null, string phone = null)
    {
        string oldName = Name;
        Name = name;
        Age = age;
        if (email != null) Email = email;
        if (phone != null) Phone = phone;
        OnPersonUpdated($"Обновлена информация о {oldName} -> {Name}");
    }
    
    // Удаление человека
    public void Delete()
    {
        AllPeople.Remove(this);
        OnPersonDeleted($"Удален {GetType().Name}: {Name}");
    }
    
    // Статические методы для работы с коллекцией
    public static List<Person> FindByName(string name)
    {
        return AllPeople.Where(p => p.Name.Contains(name, StringComparison.OrdinalIgnoreCase)).ToList();
    }
    
    public static List<Person> FindByAge(int minAge, int maxAge)
    {
        return AllPeople.Where(p => p.Age >= minAge && p.Age <= maxAge).ToList();
    }
    
    public static void DisplayAllPeople()
    {
        Console.WriteLine($"\nВсего людей: {AllPeople.Count}");
        foreach (var person in AllPeople)
        {
            Console.WriteLine($"  {person.GetInfo()}");
        }
    }
}

// ========== КЛАСС STUDENT ==========
public class Student : Person
{
    public string University { get; set; }
    public int Course { get; set; }
    public string Major { get; set; }
    private List<int> grades = new List<int>();
    public double Scholarship { get; private set; }
    
    // События для студента
    public event GradeChangeHandler GradeAdded;
    public event PersonEventHandler ExamPassed;
    
    public Student(string name, int age, string gender, string university, int course, string major) 
        : base(name, age, gender)
    {
        University = university;
        Course = course;
        Major = major;
    }
    
    public Student(string name, int age, string gender, string university, int course, string major, string email) 
        : base(name, age, gender, email)
    {
        University = university;
        Course = course;
        Major = major;
    }
    
    public void AddGrade(int grade)
    {
        if (grade >= 2 && grade <= 5)
        {
            grades.Add(grade);
            double newAverage = GetAverageGrade();
            GradeAdded?.Invoke(Name, newAverage);
            UpdateScholarship();
            Console.WriteLine($"Студенту {Name} добавлена оценка: {grade}. Средний балл: {newAverage:F2}");
        }
        else
        {
            Console.WriteLine("Оценка должна быть от 2 до 5!");
        }
    }
    
    public void AddGrades(params int[] gradesList)
    {
        foreach (var grade in gradesList)
        {
            AddGrade(grade);
        }
    }
    
    private void UpdateScholarship()
    {
        double avg = GetAverageGrade();
        double oldScholarship = Scholarship;
        if (avg >= 4.8)
            Scholarship = 5000;
        else if (avg >= 4.5)
            Scholarship = 4000;
        else if (avg >= 4.0)
            Scholarship = 3000;
        else
            Scholarship = 0;
        
        if (oldScholarship != Scholarship)
        {
            Console.WriteLine($"Стипендия {Name} изменена: {oldScholarship:C} -> {Scholarship:C}");
        }
    }
    
    public double GetAverageGrade()
    {
        if (grades.Count == 0) return 0;
        return grades.Average();
    }
    
    public string GetGradeStatus()
    {
        double avg = GetAverageGrade();
        return avg switch
        {
            >= 4.8 => "Отлично",
            >= 4.0 => "Хорошо",
            >= 3.0 => "Удовлетворительно",
            _ => "Требуется улучшение"
        };
    }
    
    public void TakeExam(string subject)
    {
        Random rand = new Random();
        int grade = rand.Next(3, 6);
        AddGrade(grade);
        ExamPassed?.Invoke($"{Name} сдал экзамен по {subject} на {grade}");
    }
    
    public void ShowGrades()
    {
        if (grades.Count > 0)
        {
            Console.Write($"Оценки {Name}: {string.Join(", ", grades)}");
            Console.WriteLine($" (Средний: {GetAverageGrade():F2}, Статус: {GetGradeStatus()})");
        }
        else
        {
            Console.WriteLine($"У {Name} пока нет оценок");
        }
    }
    
    public override string GetInfo()
    {
        return base.GetInfo() + $", Университет: {University}, Курс: {Course}, " +
               $"Специальность: {Major}, Средний балл: {GetAverageGrade():F2}, Стипендия: {Scholarship:C}";
    }
    
    public override void SayHello()
    {
        Console.WriteLine($"Привет! Я студент {Name}, учусь на {Course}-м курсе в {University} по специальности '{Major}'.");
    }
    
    public override void Work()
    {
        Console.WriteLine($"{Name} подрабатывает во время учебы.");
    }
    
    public override void Study()
    {
        Console.WriteLine($"{Name} усердно учится на {Course} курсе.");
        if (GetAverageGrade() >= 4.5)
            Console.WriteLine($"  Отличная успеваемость! Средний балл: {GetAverageGrade():F2}");
    }
    
    public override void Relax()
    {
        Console.WriteLine($"{Name} отдыхает с друзьями в студенческом кампусе.");
    }
}

// ========== КЛАСС EMPLOYEE ==========
public class Employee : Person
{
    public string Company { get; set; }
    public decimal Salary { get; set; }
    public string Position { get; set; }
    public int Experience { get; set; }
    private List<string> projects = new List<string>();
    
    // События для сотрудника
    public event SalaryChangeHandler SalaryChanged;
    public event PersonEventHandler ProjectAdded;
    
    public Employee(string name, int age, string gender, string company, decimal salary, string position, int experience) 
        : base(name, age, gender)
    {
        Company = company;
        Salary = salary;
        Position = position;
        Experience = experience;
    }
    
    public void AddProject(string project)
    {
        projects.Add(project);
        ProjectAdded?.Invoke($"{Name} добавлен в проект: {project}");
        Console.WriteLine($"{Name} добавлен в проект: {project}");
    }
    
    public void AddProjects(params string[] projectsList)
    {
        foreach (var project in projectsList)
        {
            AddProject(project);
        }
    }
    
    public void RaiseSalary(decimal amount)
    {
        if (amount > 0)
        {
            decimal oldSalary = Salary;
            Salary += amount;
            SalaryChanged?.Invoke(Name, Salary);
            Console.WriteLine($"Зарплата {Name} повышена с {oldSalary:C} до {Salary:C}");
        }
    }
    
    public void RaiseSalaryByPercent(decimal percent)
    {
        if (percent > 0)
        {
            decimal oldSalary = Salary;
            Salary += Salary * percent / 100;
            SalaryChanged?.Invoke(Name, Salary);
            Console.WriteLine($"Зарплата {Name} повышена на {percent}% с {oldSalary:C} до {Salary:C}");
        }
    }
    
    public decimal CalculateBonus()
    {
        return Salary * (Experience / 100m);
    }
    
    public IReadOnlyList<string> GetProjects() => projects.AsReadOnly();
    
    public override string GetInfo()
    {
        string projectsStr = projects.Count > 0 ? string.Join(", ", projects) : "нет проектов";
        return base.GetInfo() + $", Компания: {Company}, Должность: {Position}, " +
               $"Стаж: {Experience} лет, Зарплата: {Salary:C}, Проекты: {projectsStr}";
    }
    
    public override void SayHello()
    {
        Console.WriteLine($"Здравствуйте! Я {Name}, работаю в {Company} в должности {Position}.");
    }
    
    public override void Work()
    {
        Console.WriteLine($"{Name} работает в {Company} над проектами: {(projects.Count > 0 ? string.Join(", ", projects) : "новые проекты")}");
    }
    
    public override void Study()
    {
        Console.WriteLine($"{Name} повышает квалификацию на курсах.");
    }
    
    public override void Relax()
    {
        Console.WriteLine($"{Name} отдыхает после рабочего дня.");
    }
}

// ========== КЛАСС TEACHER (наследует от Employee) ==========
public class Teacher : Employee
{
    public string Subject { get; set; }
    public string AcademicDegree { get; set; }
    private List<Student> students = new List<Student>();
    
    // События для преподавателя
    public event PersonEventHandler StudentAdded;
    public event PersonEventHandler ExamConducted;
    
    public Teacher(string name, int age, string gender, string company, decimal salary, 
                   string position, int experience, string subject, string academicDegree) 
        : base(name, age, gender, company, salary, position, experience)
    {
        Subject = subject;
        AcademicDegree = academicDegree;
    }
    
    public void AddStudent(Student student)
    {
        if (!students.Contains(student))
        {
            students.Add(student);
            StudentAdded?.Invoke($"Студент {student.Name} добавлен к преподавателю {Name}");
            Console.WriteLine($"Студент {student.Name} добавлен к преподавателю {Name}");
        }
    }
    
    public void AddStudents(params Student[] studentsList)
    {
        foreach (var student in studentsList)
        {
            AddStudent(student);
        }
    }
    
    public void GradeStudent(Student student, int grade)
    {
        if (students.Contains(student))
        {
            student.AddGrade(grade);
            Console.WriteLine($"Преподаватель {Name} выставил оценку {grade} студенту {student.Name} по предмету {Subject}");
        }
        else
        {
            Console.WriteLine($"Студент {student.Name} не учится у преподавателя {Name}");
        }
    }
    
    public void ConductExam()
    {
        Console.WriteLine($"\n{Name} проводит экзамен по предмету {Subject}");
        ExamConducted?.Invoke($"{Name} проводит экзамен по {Subject}");
        Random rand = new Random();
        foreach (var student in students)
        {
            int grade = rand.Next(3, 6);
            GradeStudent(student, grade);
        }
    }
    
    public int StudentsCount => students.Count;
    
    public override void SayHello()
    {
        Console.WriteLine($"Здравствуйте! Я преподаватель {Name}, {AcademicDegree}, веду предмет '{Subject}'.");
    }
    
    public override string GetInfo()
    {
        return base.GetInfo() + $", Предмет: {Subject}, Ученая степень: {AcademicDegree}, Студентов: {StudentsCount}";
    }
    
    public override void Work()
    {
        base.Work();
        Console.WriteLine($"  Преподает {Subject} {StudentsCount} студентам");
    }
    
    public override void Study()
    {
        Console.WriteLine($"{Name} повышает квалификацию по предмету {Subject}.");
    }
}

// ========== КЛАСС PROFESSOR (множественное наследование через интерфейсы) ==========
public interface IResearcher
{
    int PublicationsCount { get; }
    void DoResearch();
    void PublishArticle(string title);
}

public interface IMentor
{
    int StudentsMentored { get; }
    void MentorStudent(Student student);
    void GiveAdvice();
}

public interface IConferenceSpeaker
{
    int ConferencesAttended { get; }
    void SpeakAtConference(string conferenceName);
    void PresentResearch();
}

public class Professor : Teacher, IResearcher, IMentor, IConferenceSpeaker
{
    public string ResearchField { get; set; }
    public string LaboratoryName { get; set; }
    
    private List<string> publications = new List<string>();
    private List<Student> mentoredStudents = new List<Student>();
    private List<string> conferences = new List<string>();
    
    // События для профессора
    public event PersonEventHandler ArticlePublished;
    public event PersonEventHandler ConferenceSpoke;
    
    public Professor(string name, int age, string gender, string company, decimal salary,
                     string position, int experience, string subject, string academicDegree,
                     string researchField, string laboratoryName) 
        : base(name, age, gender, company, salary, position, experience, subject, academicDegree)
    {
        ResearchField = researchField;
        LaboratoryName = laboratoryName;
    }
    
    // IResearcher
    public int PublicationsCount => publications.Count;
    
    public void DoResearch()
    {
        Console.WriteLine($"Профессор {Name} проводит исследования в области {ResearchField} в лаборатории {LaboratoryName}");
    }
    
    public void PublishArticle(string title)
    {
        publications.Add(title);
        ArticlePublished?.Invoke($"Профессор {Name} опубликовал статью: {title}");
        Console.WriteLine($"Профессор {Name} опубликовал статью: '{title}' (Всего: {PublicationsCount})");
    }
    
    // IMentor
    public int StudentsMentored => mentoredStudents.Count;
    
    public void MentorStudent(Student student)
    {
        if (!mentoredStudents.Contains(student))
        {
            mentoredStudents.Add(student);
            Console.WriteLine($"Профессор {Name} взял шефство над студентом {student.Name}");
        }
    }
    
    public void GiveAdvice()
    {
        Console.WriteLine($"Профессор {Name} дает ценный совет по научной карьере студентам");
    }
    
    // IConferenceSpeaker
    public int ConferencesAttended => conferences.Count;
    
    public void SpeakAtConference(string conferenceName)
    {
        conferences.Add(conferenceName);
        ConferenceSpoke?.Invoke($"Профессор {Name} выступил на {conferenceName}");
        Console.WriteLine($"Профессор {Name} выступает на конференции: {conferenceName}");
    }
    
    public void PresentResearch()
    {
        Console.WriteLine($"Профессор {Name} представляет результаты исследований в {ResearchField}");
    }
    
    public override void SayHello()
    {
        Console.WriteLine($"Здравствуйте! Я профессор {Name}, {AcademicDegree}, заведующий лабораторией {LaboratoryName}.");
    }
    
    public override string GetInfo()
    {
        return base.GetInfo() + $", Лаборатория: {LaboratoryName}, Область исследований: {ResearchField}, " +
               $"Публикаций: {PublicationsCount}, Студентов под наставничеством: {StudentsMentored}, " +
               $"Конференций: {ConferencesAttended}";
    }
    
    public override void Work()
    {
        base.Work();
        DoResearch();
    }
    
    public void ConductScientificConference()
    {
        Console.WriteLine($"Профессор {Name} организует научную конференцию по {ResearchField}");
    }
}

// ========== КЛАСС UNIVERSITY ДЛЯ УПРАВЛЕНИЯ КОЛЛЕКЦИЯМИ ==========
public class University
{
    public string Name { get; set; }
    private List<Student> students = new List<Student>();
    private List<Teacher> teachers = new List<Teacher>();
    private List<Employee> employees = new List<Employee>();
    
    // События университета
    public event PersonEventHandler StudentEnrolled;
    public event PersonEventHandler TeacherHired;
    public event PersonEventHandler EmployeeHired;
    
    public University(string name)
    {
        Name = name;
    }
    
    public void EnrollStudent(Student student)
    {
        students.Add(student);
        StudentEnrolled?.Invoke($"Студент {student.Name} зачислен в {Name}");
        Console.WriteLine($"Студент {student.Name} зачислен в университет {Name}");
    }
    
    public void HireTeacher(Teacher teacher)
    {
        teachers.Add(teacher);
        TeacherHired?.Invoke($"Преподаватель {teacher.Name} принят на работу в {Name}");
        Console.WriteLine($"Преподаватель {teacher.Name} принят на работу в университет {Name}");
    }
    
    public void HireEmployee(Employee employee)
    {
        employees.Add(employee);
        EmployeeHired?.Invoke($"Сотрудник {employee.Name} принят на работу в {Name}");
        Console.WriteLine($"Сотрудник {employee.Name} принят на работу в университет {Name}");
    }
    
    public List<Student> GetStudentsByCourse(int course)
    {
        return students.Where(s => s.Course == course).ToList();
    }
    
    public List<Student> GetStudentsByAverageGrade(double minGrade)
    {
        return students.Where(s => s.GetAverageGrade() >= minGrade).ToList();
    }
    
    public List<Teacher> GetTeachersBySubject(string subject)
    {
        return teachers.Where(t => t.Subject.Equals(subject, StringComparison.OrdinalIgnoreCase)).ToList();
    }
    
    public void DisplayUniversityInfo()
    {
        Console.WriteLine($"\n=== Университет '{Name}' ===");
        Console.WriteLine($"Студентов: {students.Count}");
        Console.WriteLine($"Преподавателей: {teachers.Count}");
        Console.WriteLine($"Сотрудников: {employees.Count}");
    }
    
    public void DisplayAllStudents()
    {
        Console.WriteLine($"\n--- Студенты университета '{Name}' ---");
        foreach (var student in students)
        {
            Console.WriteLine($"  {student.GetInfo()}");
        }
    }
    
    public void ConductSession()
    {
        Console.WriteLine($"\n=== Проведение сессии в университете '{Name}' ===");
        foreach (var teacher in teachers)
        {
            teacher.ConductExam();
        }
    }
    
    public void ShowStatistics()
    {
        if (students.Count == 0) return;
        
        double avgAge = students.Average(s => s.Age);
        double avgGrade = students.Average(s => s.GetAverageGrade());
        var bestStudent = students.OrderByDescending(s => s.GetAverageGrade()).FirstOrDefault();
        var worstStudent = students.OrderBy(s => s.GetAverageGrade()).FirstOrDefault();
        
        Console.WriteLine($"\n=== Статистика университета '{Name}' ===");
        Console.WriteLine($"Средний возраст студентов: {avgAge:F1} лет");
        Console.WriteLine($"Средний балл студентов: {avgGrade:F2}");
        if (bestStudent != null)
            Console.WriteLine($"Лучший студент: {bestStudent.Name} (средний балл: {bestStudent.GetAverageGrade():F2})");
        if (worstStudent != null)
            Console.WriteLine($"Студент с наименьшим баллом: {worstStudent.Name} (средний балл: {worstStudent.GetAverageGrade():F2})");
    }
}

// ========== ДЕМОНСТРАЦИЯ ==========

Console.WriteLine("========== ИНДИВИДУАЛЬНЫЙ ПРОЕКТ: СИСТЕМА УПРАВЛЕНИЯ УНИВЕРСИТЕТОМ ==========");
Console.WriteLine("Коллекции, делегаты и события в C#\n");

// ========== 1. СОЗДАНИЕ ОБЪЕКТОВ ==========
Console.WriteLine("=== 1. СОЗДАНИЕ ОБЪЕКТОВ ===\n");

// Подписка на глобальные события Person
Person.PersonCreated += (msg) => Console.WriteLine($"[ГЛОБАЛЬНОЕ СОБЫТИЕ] {msg}");

// Создание студентов
Student student1 = new Student("Анна Смирнова", 20, "Женский", "МГУ", 3, "Программирование", "anna@mail.ru");
Student student2 = new Student("Олег Новиков", 22, "Мужской", "МГУ", 4, "Искусственный интеллект");
Student student3 = new Student("Елена Попова", 19, "Женский", "МГУ", 2, "Биоинформатика");
Student student4 = new Student("Дмитрий Козлов", 21, "Мужской", "МГУ", 3, "Программирование");

// Создание сотрудников
Employee employee = new Employee("Пётр Сидоров", 35, "Мужской", "ООО Ромашка", 75000, "Разработчик", 5);

// Создание преподавателей
Teacher teacher = new Teacher("Елена Иванова", 45, "Женский", "МГУ", 120000, "Доцент", 15, "Математика", "Кандидат наук");

// Создание профессора
Professor professor = new Professor("Сергей Николаев", 50, "Мужской", "МГУ", 200000, "Профессор", 25,
                                    "Программирование", "Доктор наук", "Искусственный интеллект", "Лаборатория ИИ");

// ========== 2. ДЕМОНСТРАЦИЯ КОЛЛЕКЦИЙ ==========
Console.WriteLine("\n=== 2. РАБОТА С КОЛЛЕКЦИЯМИ ===\n");

// Создание университета
University msu = new University("МГУ им. Ломоносова");

// Подписка на события университета
msu.StudentEnrolled += (msg) => Console.WriteLine($"[УНИВЕРСИТЕТ] {msg}");
msu.TeacherHired += (msg) => Console.WriteLine($"[УНИВЕРСИТЕТ] {msg}");

// Добавление людей в университет
msu.EnrollStudent(student1);
msu.EnrollStudent(student2);
msu.EnrollStudent(student3);
msu.EnrollStudent(student4);
msu.HireTeacher(teacher);
msu.HireTeacher(professor);
msu.HireEmployee(employee);

// Подписка на события студентов
student1.GradeAdded += (name, avg) => Console.WriteLine($"[СОБЫТИЕ] У студента {name} новый средний балл: {avg:F2}");
student1.ExamPassed += (msg) => Console.WriteLine($"[СОБЫТИЕ] {msg}");

// Подписка на события преподавателя
teacher.StudentAdded += (msg) => Console.WriteLine($"[СОБЫТИЕ] {msg}");
teacher.ExamConducted += (msg) => Console.WriteLine($"[СОБЫТИЕ] {msg}");

// Подписка на события профессора
professor.ArticlePublished += (msg) => Console.WriteLine($"[СОБЫТИЕ] {msg}");
professor.ConferenceSpoke += (msg) => Console.WriteLine($"[СОБЫТИЕ] {msg}");

// Подписка на события сотрудника
employee.SalaryChanged += (name, salary) => Console.WriteLine($"[СОБЫТИЕ] У {name} новая зарплата: {salary:C}");
employee.ProjectAdded += (msg) => Console.WriteLine($"[СОБЫТИЕ] {msg}");

// ========== 3. ДЕМОНСТРАЦИЯ РАБОТЫ СТУДЕНТОВ ==========
Console.WriteLine("\n=== 3. ДЕМОНСТРАЦИЯ РАБОТЫ СТУДЕНТОВ ===\n");

student1.AddGrades(5, 4, 5, 5, 4);
student1.ShowGrades();
student1.TakeExam("Программирование");
student1.Study();

student2.AddGrade(3);
student2.AddGrade(4);
student2.ShowGrades();

// ========== 4. ДЕМОНСТРАЦИЯ РАБОТЫ ПРЕПОДАВАТЕЛЯ ==========
Console.WriteLine("\n=== 4. ДЕМОНСТРАЦИЯ РАБОТЫ ПРЕПОДАВАТЕЛЯ ===\n");

teacher.AddStudents(student1, student2);
teacher.ConductExam();

// ========== 5. ДЕМОНСТРАЦИЯ РАБОТЫ ПРОФЕССОРА ==========
Console.WriteLine("\n=== 5. ДЕМОНСТРАЦИЯ РАБОТЫ ПРОФЕССОРА ===\n");

professor.AddStudent(student3);
professor.MentorStudent(student3);
professor.DoResearch();
professor.PublishArticle("Новые методы машинного обучения");
professor.PublishArticle("Оптимизация нейросетей");
professor.SpeakAtConference("Международная конференция по ИИ 2024");
professor.PresentResearch();

// ========== 6. ДЕМОНСТРАЦИЯ РАБОТЫ СОТРУДНИКА ==========
Console.WriteLine("\n=== 6. ДЕМОНСТРАЦИЯ РАБОТЫ СОТРУДНИКА ===\n");

employee.AddProjects("Разработка ПО", "Мобильное приложение", "База данных");
employee.RaiseSalary(15000);
employee.RaiseSalaryByPercent(10);
employee.Work();

// ========== 7. РАБОТА С КОЛЛЕКЦИЯМИ LINQ ==========
Console.WriteLine("\n=== 7. РАБОТА С КОЛЛЕКЦИЯМИ LINQ ===\n");

// Фильтрация студентов по курсу
var thirdCourseStudents = msu.GetStudentsByCourse(3);
Console.WriteLine("Студенты 3-го курса:");
foreach (var s in thirdCourseStudents)
{
    Console.WriteLine($"  {s.Name}");
}

// Фильтрация студентов по успеваемости
var goodStudents = msu.GetStudentsByAverageGrade(4.0);
Console.WriteLine("\nСтуденты с хорошей успеваемостью (ср. балл >= 4.0):");
foreach (var s in goodStudents)
{
    Console.WriteLine($"  {s.Name} - {s.GetAverageGrade():F2}");
}

// Сортировка студентов по среднему баллу
var sortedByGrade = Person.AllPeople.OfType<Student>()
    .OrderByDescending(s => s.GetAverageGrade()).ToList();
Console.WriteLine("\nСтуденты по успеваемости (от лучших к худшим):");
foreach (var s in sortedByGrade)
{
    Console.WriteLine($"  {s.Name} - {s.GetAverageGrade():F2}");
}

// Группировка по курсам
var groupedByCourse = Person.AllPeople.OfType<Student>()
    .GroupBy(s => s.Course)
    .OrderBy(g => g.Key);
Console.WriteLine("\nГруппировка студентов по курсам:");
foreach (var group in groupedByCourse)
{
    Console.WriteLine($"  {group.Key} курс: {group.Count()} студентов");
    foreach (var s in group)
    {
        Console.WriteLine($"    - {s.Name}");
    }
}

// ========== 8. СТАТИСТИКА УНИВЕРСИТЕТА ==========
msu.DisplayUniversityInfo();
msu.DisplayAllStudents();
msu.ShowStatistics();

// ========== 9. ДЕМОНСТРАЦИЯ ПОИСКА ==========
Console.WriteLine("\n=== 9. ПОИСК В КОЛЛЕКЦИЯХ ===\n");

var foundPeople = Person.FindByName("Анна");
Console.WriteLine("Найденные люди по имени 'Анна':");
foreach (var p in foundPeople)
{
    Console.WriteLine($"  {p.GetInfo()}");
}

var youngPeople = Person.FindByAge(18, 22);
Console.WriteLine($"\nМолодые люди (18-22 года): {youngPeople.Count}");

// ========== 10. ДЕМОНСТРАЦИЯ УДАЛЕНИЯ ==========
Console.WriteLine("\n=== 10. ДЕМОНСТРАЦИЯ УДАЛЕНИЯ ===\n");

Person.DisplayAllPeople();

Console.WriteLine("\nУдаляем студента Олега Новикова...");
student2.Delete();

Person.DisplayAllPeople();

// ========== 11. ДЕМОНСТРАЦИЯ ЛЯМБДА-ВЫРАЖЕНИЙ ==========
Console.WriteLine("\n=== 11. ДЕМОНСТРАЦИЯ ЛЯМБДА-ВЫРАЖЕНИЙ ===\n");

// Использование лямбда-выражений для фильтрации
var excellentStudents = Person.AllPeople.OfType<Student>()
    .Where(s => s.GetAverageGrade() >= 4.5)
    .Select(s => new { Name = s.Name, AverageGrade = s.GetAverageGrade() });

Console.WriteLine("Отличники (ср. балл >= 4.5):");
foreach (var s in excellentStudents)
{
    Console.WriteLine($"  {s.Name} - {s.AverageGrade:F2}");
}

// Использование лямбда-выражений для агрегации
double totalScholarship = Person.AllPeople.OfType<Student>().Sum(s => s.Scholarship);
Console.WriteLine($"\nОбщая сумма стипендий: {totalScholarship:C}");

// Использование Func делегата
Func<Student, string> getStudentInfo = s => $"{s.Name} ({s.Course} курс, {s.GetAverageGrade():F2})";
Console.WriteLine("\nИнформация о студентах через Func делегат:");
foreach (var s in Person.AllPeople.OfType<Student>().Take(3))
{
    Console.WriteLine($"  {getStudentInfo(s)}");
}

// ========== 12. ФИНАЛЬНАЯ СТАТИСТИКА ==========
Console.WriteLine("\n=== 12. ФИНАЛЬНАЯ СТАТИСТИКА ===\n");
Console.WriteLine($"Всего создано людей: {Person.AllPeople.Count}");
Console.WriteLine($"Из них студентов: {Person.AllPeople.OfType<Student>().Count()}");
Console.WriteLine($"Преподавателей: {Person.AllPeople.OfType<Teacher>().Count()}");
Console.WriteLine($"Сотрудников: {Person.AllPeople.OfType<Employee>().Count()}");
Console.WriteLine($"Профессоров: {Person.AllPeople.OfType<Professor>().Count()}");

Console.WriteLine("\n========== ИНДИВИДУАЛЬНЫЙ ПРОЕКТ ЗАВЕРШЕН ==========");


========== ИНДИВИДУАЛЬНЫЙ ПРОЕКТ: СИСТЕМА УПРАВЛЕНИЯ УНИВЕРСИТЕТОМ ==========
Коллекции, делегаты и события в C#

=== 1. СОЗДАНИЕ ОБЪЕКТОВ ===

[ГЛОБАЛЬНОЕ СОБЫТИЕ] Создан Student: Анна Смирнова
[ГЛОБАЛЬНОЕ СОБЫТИЕ] Создан Student: Олег Новиков
[ГЛОБАЛЬНОЕ СОБЫТИЕ] Создан Student: Елена Попова
[ГЛОБАЛЬНОЕ СОБЫТИЕ] Создан Student: Дмитрий Козлов
[ГЛОБАЛЬНОЕ СОБЫТИЕ] Создан Employee: Пётр Сидоров
[ГЛОБАЛЬНОЕ СОБЫТИЕ] Создан Teacher: Елена Иванова
[ГЛОБАЛЬНОЕ СОБЫТИЕ] Создан Professor: Сергей Николаев

=== 2. РАБОТА С КОЛЛЕКЦИЯМИ ===

[УНИВЕРСИТЕТ] Студент Анна Смирнова зачислен в МГУ им. Ломоносова
Студент Анна Смирнова зачислен в университет МГУ им. Ломоносова
[УНИВЕРСИТЕТ] Студент Олег Новиков зачислен в МГУ им. Ломоносова
Студент Олег Новиков зачислен в университет МГУ им. Ломоносова
[УНИВЕРСИТЕТ] Студент Елена Попова зачислен в МГУ им. Ломоносова
Студент Елена Попова зачислен в университет МГУ им. Ломоносова
[УНИВЕРСИТЕТ] Студент Дмитрий Козлов зачислен в МГУ им. Ломоносова
Студент Д